# Strong baseline on Colab: RoBERTa-base, current utterance only

Milestone 2, condition B (see `strong-baseline.md`). Run the cells from top to bottom.

1. **Runtime → Change runtime type → T4 GPU** (or any GPU), then run the first cell to check it.
2. Each full run (one seed) takes about 4–5 minutes on a T4 or L4 GPU, plus a few minutes of setup the first time.
3. **Save the results before closing Colab** (last section): files on the Colab machine are deleted when the session ends.

In [ ]:
# 1. Check that a GPU is attached
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Get the code

In [ ]:
BRANCH = "m2-strong-baseline"   # branch with the strong-baseline code
REPO = "https://github.com/3XIA00/CIS5300_Final_Project.git"

import os
if not os.path.isdir("/content/CIS5300_Final_Project"):
    !git clone -q -b {BRANCH} {REPO} /content/CIS5300_Final_Project
%cd /content/CIS5300_Final_Project
!git pull -q origin {BRANCH}
!git log --oneline -1

In [ ]:
# Colab already has torch; this installs or updates transformers if needed
!pip install -q -r strong_baseline/requirements.txt

## 3. Smoke test (about a minute)

Trains for one epoch on 64 utterances per split, just to check the whole path works. Its scores mean nothing.
It downloads `roberta-base` (about 500 MB) the first time.

In [ ]:
!python -m strong_baseline.train --seed 13 --limit 64 --epochs 1 --fp16 --no-checkpoint --output-dir /content/smoke

## 4. Full runs

One run per seed. Plan seeds: 13, 42, 2026, 31415, 27182 (all five are reported in strong-baseline.md).
If two people share the work, give each person different seeds.

In [ ]:
SEEDS = [13, 42, 2026]

for seed in SEEDS:
    print(f"===== seed {seed} =====")
    !python -m strong_baseline.train --seed {seed} --fp16

## 5. Results table

In [ ]:
import json, statistics
from pathlib import Path

rows = []
for path in sorted(Path("output/roberta-base").glob("seed*/run.json")):
    run = json.loads(path.read_text())
    if run.get("limit"):
        continue  # skip smoke tests
    r = run["results"]
    rows.append({
        "run": path.parent.name,
        "dev W-F1": r["dev"]["weighted_f1"], "dev M-F1": r["dev"]["macro_f1"],
        "test W-F1": r["test"]["weighted_f1"], "test M-F1": r["test"]["macro_f1"],
        "test excl. W-F1": r["test_excluding_suspected_overlap"]["weighted_f1"],
        "test excl. M-F1": r["test_excluding_suspected_overlap"]["macro_f1"],
        "best epoch": run["best_epoch"],
        "minutes": round(run["train_seconds"] / 60, 1),
        "peak GPU MB": run["peak_gpu_memory_mb"],
        "GPU": run["device"],
    })

header = list(rows[0]) if rows else []
lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
lines += ["| " + " | ".join(str(row[h]) for h in header) + " |" for row in rows]
if len(rows) > 1:  # mean and sample standard deviation (ddof=1), as in experiment_plan.md
    score_cols = [h for h in header if "F1" in h]
    cells = [f"{statistics.mean(r[h] for r in rows):.2f} ± {statistics.stdev(r[h] for r in rows):.2f}" if h in score_cols else ""
             for h in header]
    cells[0] = "mean ± sd"
    lines.append("| " + " | ".join(cells) + " |")
print("\n".join(lines))   # paste this into the Results section of strong-baseline.md

## 6. Save the results (do this before closing Colab)

Keep `output/roberta-base/` (predictions + `run.json`, small). Do **not** commit `checkpoints/` (about 500 MB per seed).

Most people should use **Option A** or **B** and send the results to one teammate, who commits them. Only one person needs to push.

**Option A: download a zip** and send it to the teammate who commits it.

In [ ]:
!zip -qr /content/roberta-base-output.zip output/roberta-base
from google.colab import files
files.download("/content/roberta-base-output.zip")

**Option B: copy to Google Drive** (also keeps the checkpoints, if you want them later).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/CIS5300
!cp -r output/roberta-base /content/drive/MyDrive/CIS5300/
# !cp -r checkpoints/roberta-base /content/drive/MyDrive/CIS5300/checkpoints   # optional, about 500 MB per seed

**Option C: push to the branch from Colab** (only if you are the one person pushing).

- Use a **fine-grained** personal access token: GitHub → Settings → Developer settings → Fine-grained tokens, repository access *only* `3XIA00/CIS5300_Final_Project`, permission *Contents: Read and write*, expiry of a few days. Do not use a classic token with `repo` scope.
- Do not paste the token into a cell or save it in the notebook. The cell below asks for it, passes it to git through an environment variable (never on the command line), hides it in any error message, and deletes it afterwards.
- It pulls the latest branch first, so your commit goes on top of anything others pushed. If the pull fails, nothing is pushed: use Option A instead.
- **If a token ever appears in any output, revoke it immediately** in GitHub settings; deleting the output is not enough.

In [ ]:
import getpass, os, stat, subprocess, tempfile

user = input("GitHub username: ").strip()
token = getpass.getpass("Fine-grained GitHub token for this repo only (not saved): ").strip()
if not token:
    raise SystemExit("no token entered; nothing pushed")
hide = lambda text: text.replace(token, "***")
askpass = None
try:
    subprocess.run(["git", "config", "user.name", user], check=True)
    subprocess.run(["git", "config", "user.email", f"{user}@users.noreply.github.com"], check=True)
    subprocess.run(["git", "add", "output/roberta-base"], check=True)
    subprocess.run(["git", "commit", "-q", "-m", "Add roberta-base strong baseline predictions and run logs"])  # no-op if nothing new

    pull = subprocess.run(["git", "pull", "--rebase", "-q", "origin", BRANCH], capture_output=True, text=True)
    if pull.returncode != 0:
        subprocess.run(["git", "rebase", "--abort"], capture_output=True)
        print("pull failed, nothing pushed; use Option A (zip) instead.\n" + hide(pull.stderr))
    else:
        # git asks this helper for the username and password; it reads them from the environment,
        # so the token never appears in a command line, a remote URL or the helper file itself.
        fd, askpass = tempfile.mkstemp(suffix=".sh")
        with os.fdopen(fd, "w") as helper:
            helper.write('#!/bin/sh\ncase "$1" in Username*) echo "$GIT_PUSH_USER" ;; *) echo "$GIT_PUSH_TOKEN" ;; esac\n')
        os.chmod(askpass, stat.S_IRWXU)
        env = dict(os.environ, GIT_ASKPASS=askpass, GIT_TERMINAL_PROMPT="0",
                   GIT_PUSH_USER=user, GIT_PUSH_TOKEN=token)
        push = subprocess.run(["git", "-c", "credential.helper=", "push",
                               "https://github.com/3XIA00/CIS5300_Final_Project.git", f"HEAD:{BRANCH}"],
                              env=env, capture_output=True, text=True)
        print("pushed" if push.returncode == 0 else "push failed, nothing pushed:\n" + hide(push.stderr))
finally:
    if askpass and os.path.exists(askpass):
        os.remove(askpass)
    del token, hide